In [24]:
# 1

import re
from collections import Counter
from pathlib import Path

import pandas as pd
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_PATH = BASE_DIR / "food_delivery_reviews.csv"

print('Training data file:', DATA_PATH.name, '| exists:', DATA_PATH.exists())

Training data file: food_delivery_reviews.csv | exists: True


In [25]:
# 2

from IPython.display import display

df = pd.read_csv(DATA_PATH)
print("Shape (rows, columns):", df.shape)
print("Missing values per column:\n", df.isna().sum().to_string())
print("\nDuplicate review texts:", df["review_text"].duplicated().sum())
display(df.head(8))

Shape (rows, columns): (320, 5)
Missing values per column:
 review_id         0
review_text       0
sentiment         0
issue_category    0
source            0

Duplicate review texts: 0


,review_id,review_text,sentiment,issue_category,source
0,1,Very disappointed with the overall service and...,Negative,General,handwritten
1,2,Completely unacceptable. The delivery guy refu...,Negative,Delivery,generated
2,3,Terrible experience. Hidden charges made the b...,Negative,General,generated
3,4,Completely unacceptable. Order was delivered v...,Negative,Delivery,generated
4,5,I order every week and the service is reliable...,Positive,General,generated
5,6,The app crashed at checkout. Really unhappy wi...,Negative,App,generated
6,7,"Perfect, will order again. Live tracking in th...",Positive,App,generated
7,8,"Excellent, thank you! Support team was polite ...",Positive,General,generated


In [26]:
print("Sentiment distribution")
display(df["sentiment"].value_counts().to_frame("reviews"))
print("Issue-category distribution")
display(df["issue_category"].value_counts().to_frame("reviews"))
print("Sentiment x category (every combination has examples)")
display(pd.crosstab(df["issue_category"], df["sentiment"], margins=True))
print("Source of the rows")
display(df["source"].value_counts().to_frame("reviews"))

Sentiment distribution


,reviews
sentiment,
Negative,160
Positive,160


Issue-category distribution


,reviews
issue_category,
General,80
Delivery,80
App,80
Food Quality,80


Sentiment x category (every combination has examples)


sentiment,Negative,Positive,All
issue_category,,,
App,40,40,80
Delivery,40,40,80
Food Quality,40,40,80
General,40,40,80
All,160,160,320


Source of the rows


,reviews
source,
generated,240
handwritten,80


In [27]:
df["n_words"] = df["review_text"].str.split().str.len()
print("Review length (words)")
display(df["n_words"].describe().round(1).to_frame("value").T)

print("Two example reviews per issue category")
display(df.groupby("issue_category").head(2).sort_values("issue_category")[["review_text", "sentiment", "issue_category"]])

Review length (words)


,count,mean,std,min,25%,50%,75%,max
value,320.0,10.1,1.7,6.0,9.0,10.0,11.0,14.0


Two example reviews per issue category


,review_text,sentiment,issue_category
5,The app crashed at checkout. Really unhappy wi...,Negative,App
6,"Perfect, will order again. Live tracking in th...",Positive,App
1,Completely unacceptable. The delivery guy refu...,Negative,Delivery
3,Completely unacceptable. Order was delivered v...,Negative,Delivery
17,Highly recommended. Portions were generous and...,Positive,Food Quality
18,Never ordering again. The pizza was burnt and ...,Negative,Food Quality
0,Very disappointed with the overall service and...,Negative,General
2,Terrible experience. Hidden charges made the b...,Negative,General


In [28]:
# 3

NEGATIONS = {"no", "not", "nor", "never", "cannot", "nothing", "none", "nobody", "neither"}
STOPWORDS = set(ENGLISH_STOP_WORDS) - NEGATIONS
STEMMER = PorterStemmer()


def clean_tokens(text):
    """Steps 1-5: returns the content words (lowercased, no punctuation, no stopwords)."""
    text = str(text).lower()                                  # 1. lowercase
    text = text.replace("can't", "cannot").replace("won't", "will not")
    text = re.sub(r"n't\b", " not", text)                     # 2. didn't -> did not
    text = re.sub(r"'(s|m|re|ve|ll|d)\b", "", text)           #    it's -> it, I'm -> I
    text = re.sub(r"[^a-z\s]", " ", text)                     # 3. drop punctuation, digits, emojis
    tokens = text.split()                                     # 4. tokenise
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]   # 5. stopword removal


def preprocess(text):
    """Full pipeline: returns (content_words, stemmed_text)."""
    words = clean_tokens(text)
    stems = [STEMMER.stem(w) for w in words]                  # 6. stemming
    return words, " ".join(stems)

In [29]:
# Step-by-step trace of one noisy review
sample = "The rider DIDN'T come!!! Waited 90 mins  - food was cold, it's unacceptable."
t = sample.lower();                                              print("1 lowercase     :", t)
t = t.replace("can't", "cannot").replace("won't", "will not")
t = re.sub(r"n't\b", " not", t); t = re.sub(r"'(s|m|re|ve|ll|d)\b", "", t);  print("2 contractions  :", t)
t = re.sub(r"[^a-z\s]", " ", t);                                 print("3 punct/num/emoji:", " ".join(t.split()))
tokens = t.split();                                              print("4 tokens        :", tokens)
tokens = [w for w in tokens if w not in STOPWORDS and len(w) > 1]; print("5 no stopwords  :", tokens)
print("6 stemmed       :", [STEMMER.stem(w) for w in tokens])

1 lowercase     : the rider didn't come!!! waited 90 mins  - food was cold, it's unacceptable.
2 contractions  : the rider did not come!!! waited 90 mins  - food was cold, it unacceptable.
3 punct/num/emoji: the rider did not come waited mins food was cold it unacceptable
4 tokens        : ['the', 'rider', 'did', 'not', 'come', 'waited', 'mins', 'food', 'was', 'cold', 'it', 'unacceptable']
5 no stopwords  : ['rider', 'did', 'not', 'come', 'waited', 'mins', 'food', 'cold', 'unacceptable']
6 stemmed       : ['rider', 'did', 'not', 'come', 'wait', 'min', 'food', 'cold', 'unaccept']


In [30]:
examples = [
    "Delivery was SUPER fast!!! ",
    "The app isn't working, payment failed 3 times.",
    "Food was not good, and it arrived cold.",
    "Can't complain - great value for money :)",
]
rows = []
for text in examples:
    words, stemmed = preprocess(text)
    rows.append({"original": text, "content words": " ".join(words), "stored (stemmed)": stemmed})
display(pd.DataFrame(rows))

,original,content words,stored (stemmed)
0,Delivery was SUPER fast!!!,delivery super fast,deliveri super fast
1,"The app isn't working, payment failed 3 times.",app not working payment failed times,app not work payment fail time
2,"Food was not good, and it arrived cold.",food not good arrived cold,food not good arriv cold
3,Can't complain - great value for money :),cannot complain great value money,cannot complain great valu money


In [31]:
# 4

REQUIRED_COLUMNS = {"review_text", "sentiment", "issue_category"}


def make_sentiment_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, C=10)),
    ])


def make_category_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", MultinomialNB(alpha=0.1)),
    ])


def load_training_data(path):
    """Read the labelled CSV. Returns a DataFrame, or None if the file is missing/unreadable."""
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError):
        return None

In [32]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

df["processed"] = df["review_text"].map(lambda t: preprocess(t)[1])
strata = df["sentiment"] + "|" + df["issue_category"]          # keep all 8 combinations balanced in the split
X_train, X_test, train_df, test_df = train_test_split(
    df["processed"], df, test_size=0.2, random_state=42, stratify=strata)
print(f"Train: {len(X_train)} reviews | Test: {len(X_test)} reviews")

def pipe(clf):
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", clf)])

candidates = {
    "Logistic Regression": lambda: pipe(LogisticRegression(max_iter=1000, C=10)),
    "Naive Bayes":         lambda: pipe(MultinomialNB(alpha=0.1)),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
rows = []
for target in ["sentiment", "issue_category"]:
    for name, make in candidates.items():
        model = make().fit(X_train, train_df[target])
        test_acc = accuracy_score(test_df[target], model.predict(X_test))
        cv_acc = cross_val_score(make(), df["processed"], df[target], cv=cv).mean()
        rows.append({"task": target, "model": name, "test accuracy": round(test_acc, 3), "5-fold CV accuracy": round(cv_acc, 3)})
display(pd.DataFrame(rows))

Train: 256 reviews | Test: 64 reviews


,task,model,test accuracy,5-fold CV accuracy
0,sentiment,Logistic Regression,0.953,0.962
1,sentiment,Naive Bayes,0.953,0.972
2,issue_category,Logistic Regression,1.000,0.994
3,issue_category,Naive Bayes,1.000,0.981


In [33]:
sent_model = make_sentiment_model().fit(X_train, train_df["sentiment"])
cat_model = make_category_model().fit(X_train, train_df["issue_category"])
print("=== Sentiment (Logistic Regression) - test set ===")
print(classification_report(test_df["sentiment"], sent_model.predict(X_test)))
print("=== Issue category (Naive Bayes) - test set ===")
print(classification_report(test_df["issue_category"], cat_model.predict(X_test)))

=== Sentiment (Logistic Regression) - test set ===
              precision    recall  f1-score   support

    Negative       0.97      0.94      0.95        32
    Positive       0.94      0.97      0.95        32

    accuracy                           0.95        64
   macro avg       0.95      0.95      0.95        64
weighted avg       0.95      0.95      0.95        64

=== Issue category (Naive Bayes) - test set ===
              precision    recall  f1-score   support

         App       1.00      1.00      1.00        16
    Delivery       1.00      1.00      1.00        16
Food Quality       1.00      1.00      1.00        16
     General       1.00      1.00      1.00        16

    accuracy                           1.00        64
   macro avg       1.00      1.00      1.00        64
weighted avg       1.00      1.00      1.00        64



In [42]:
import re
from collections import Counter

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SENTIMENTS = ["Positive", "Negative"]

CATEGORIES = [
    "Delivery",
    "Food Quality",
    "App",
    "General"
]

MIN_TRAIN_SAMPLES = 10

REQUIRED_COLUMNS = {
    "review_text",
    "sentiment",
    "issue_category"
}

def preprocess(text):
    """
    Clean the review text and return:
    1. words
    2. cleaned text
    """

    if text is None:
        return [], ""

    text = str(text).lower().strip()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Keep only letters and spaces
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Split into words
    words = text.split()

    # Remove very short words
    words = [word for word in words if len(word) > 1]

    # Clean text for ML model
    cleaned_text = " ".join(words)

    return words, cleaned_text



def make_sentiment_model():
    """
    Create the sentiment classification model.
    """

    return Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=1
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000
            )
        )
    ])


def make_category_model():
    """
    Create the issue-category classification model.
    """

    return Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=1
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000
            )
        )
    ])


class InvalidReview(ValueError):
    """The review text is empty or contains no usable words."""


class ClassifierUnavailable(RuntimeError):
    """Not enough labelled training data to classify yet."""



class ReviewIntelligenceSystem:

    def __init__(
        self,
        train_df=None,
        min_samples=MIN_TRAIN_SAMPLES
    ):

        self.min_samples = min_samples

        # Store reviews here
        self.reviews = []

        # Model status
        self.ready = False
        self.status = ""

        # Train models
        self.train(train_df)



    def train(self, df):

        """
        Train sentiment and category models.
        """

        self.ready = False

        if df is None:

            self.status = (
                "Training data could not be loaded."
            )

            return


        if not REQUIRED_COLUMNS.issubset(df.columns):

            missing = REQUIRED_COLUMNS - set(df.columns)

            self.status = (
                f"Missing required columns: {sorted(missing)}"
            )

            return

        df = df.dropna(
            subset=list(REQUIRED_COLUMNS)
        )

        if len(df) < self.min_samples:

            self.status = (
                f"Only {len(df)} labelled training samples found; "
                f"at least {self.min_samples} are needed "
                f"to classify reviews."
            )

            return

        if df["sentiment"].nunique() < 2:

            self.status = (
                "Training data needs at least "
                "2 different sentiment labels."
            )

            return

        if df["issue_category"].nunique() < 2:

            self.status = (
                "Training data needs at least "
                "2 different issue categories."
            )

            return

        processed_reviews = []

        valid_rows = []

        for index, text in df["review_text"].items():

            words, cleaned_text = preprocess(text)

            if cleaned_text:

                processed_reviews.append(cleaned_text)
                valid_rows.append(index)

        if len(processed_reviews) < self.min_samples:

            self.status = (
                f"Only {len(processed_reviews)} usable reviews found; "
                f"at least {self.min_samples} are needed."
            )

            return

        df = df.loc[valid_rows].copy()

        self.sentiment_model = make_sentiment_model()

        self.sentiment_model.fit(
            processed_reviews,
            df["sentiment"]
        )

        self.category_model = make_category_model()

        self.category_model.fit(
            processed_reviews,
            df["issue_category"]
        )


        # Store training information
        self.n_train = len(df)

        self.ready = True

        self.status = (
            f"Models trained on {len(df)} labelled reviews."
        )

    def classify(self, raw_text):

        """
        Classify a new review.

        Returns:
        - sentiment
        - sentiment confidence
        - category
        - category confidence
        """

        words, cleaned_text = preprocess(raw_text)

        if not cleaned_text:

            raise InvalidReview(
                "The review has no usable words."
            )

        if not self.ready:

            raise ClassifierUnavailable(
                self.status
            )

        sentiment_probs = (
            self.sentiment_model
            .predict_proba([cleaned_text])[0]
        )

        category_probs = (
            self.category_model
            .predict_proba([cleaned_text])[0]
        )

        sentiment_index = sentiment_probs.argmax()

        category_index = category_probs.argmax()

        return {

            "sentiment":
                self.sentiment_model
                .classes_[sentiment_index],

            "sentiment_conf":
                float(sentiment_probs.max()),

            "category":
                self.category_model
                .classes_[category_index],

            "category_conf":
                float(category_probs.max())
        }

    def add_review(self, raw_text):

        """
        Preprocess, classify and store a review.
        """

        words, cleaned_text = preprocess(raw_text)

        if not cleaned_text:

            raise InvalidReview(
                "The review has no usable words."
            )

        record = {

            "id":
                len(self.reviews) + 1,

            "raw":
                str(raw_text).strip(),

            "words":
                words,

            "stemmed":
                cleaned_text,

            "sentiment":
                "Unclassified",

            "category":
                "Unclassified"
        }

        if self.ready:

            result = self.classify(raw_text)


            record["sentiment"] = (
                result["sentiment"]
            )

            record["category"] = (
                result["category"]
            )

            record["sentiment_conf"] = (
                result["sentiment_conf"]
            )

            record["category_conf"] = (
                result["category_conf"]
            )

        self.reviews.append(record)


        return record

    def summary(self, top_n=5):

        """
        Generate a summary of stored reviews.
        """

        # Sentiment count
        sentiment_count = Counter(
            r["sentiment"]
            for r in self.reviews
        )


        # Category count
        category_count = Counter(
            r["category"]
            for r in self.reviews
        )


        # Word count
        word_count = Counter(
            word
            for r in self.reviews
            for word in r["words"]
        )


        return {

            "total":
                len(self.reviews),

            "sentiments": {
                label:
                    sentiment_count.get(label, 0)

                for label in (
                    SENTIMENTS +
                    (
                        ["Unclassified"]
                        if sentiment_count.get("Unclassified")
                        else []
                    )
                )
            },

            "categories": {
                label:
                    category_count.get(label, 0)

                for label in (
                    CATEGORIES +
                    (
                        ["Unclassified"]
                        if category_count.get("Unclassified")
                        else []
                    )
                )
            },

            "top_words":
                word_count.most_common(top_n)
        }

In [43]:

def show_result(res):
    print(f"   Sentiment     : {res['sentiment']}  (confidence {res['sentiment_conf']:.0%})")
    print(f"   Issue category: {res['category']}  (confidence {res['category_conf']:.0%})")


def option_add(system, input_fn):
    text = input_fn("Enter the customer review: ").strip()
    try:
        rec = system.add_review(text)
    except InvalidReview as e:
        print(f"   Could not add review: {e}")
        return
    print(f"   Review #{rec['id']} added.")
    print(f"   Stored (pre-processed): {rec['stemmed']}")
    if system.ready:
        show_result(rec)
    else:
        print(f"   Note: not classified yet. {system.status}")


def option_classify(system, input_fn):
    text = input_fn("Enter review text (or a stored review number): ").strip()
    if text.isdigit():                                    # classify an already-stored review
        idx = int(text)
        if not 1 <= idx <= len(system.reviews):
            print(f"   No stored review with number {idx}. Stored reviews: {len(system.reviews)}.")
            return
        text = system.reviews[idx - 1]["raw"]
        print(f"   Review #{idx}: {text}")
    try:
        show_result(system.classify(text))
    except InvalidReview as e:
        print(f"   Cannot classify: {e}")
    except ClassifierUnavailable as e:
        print(f"   Classification is not available. {e}")


def option_report(system):
    s = system.summary()
    print("\n" + "=" * 46)
    print("  REVIEW SUMMARY REPORT")
    print("=" * 46)
    print(f"Total reviews added: {s['total']}")
    if s["total"] == 0:
        print("No reviews yet. Use option 1 to add some.")
        return
    print("\nBy sentiment:")
    for k, v in s["sentiments"].items():
        print(f"   {k:<14}{v}")
    print("\nBy issue category:")
    for k, v in s["categories"].items():
        print(f"   {k:<14}{v}")
    print("\nTop 5 most frequent content words:")
    for rank, (w, n) in enumerate(s["top_words"], start=1):
        print(f"   {rank}. {w:<14}{n}")


def main(system=None, input_fn=input):
    if system is None:
        system = ReviewIntelligenceSystem(load_training_data(DATA_PATH))
    print(f"[{system.status}]")
    while True:
        print("\n=== Food Delivery Review Intelligence System ===")
        print("1. Add a new review")
        print("2. Classify a review")
        print("3. View summary report")
        print("4. Exit")
        try:
            choice = input_fn("Choose an option (1-4): ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nInput closed. Goodbye!")
            break
        if choice == "1":
            option_add(system, input_fn)
        elif choice == "2":
            option_classify(system, input_fn)
        elif choice == "3":
            option_report(system)
        elif choice == "4":
            print("Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number from 1 to 4.")

In [44]:
# 6

def scripted(answers):
    """Return an input() replacement that replays `answers` and echoes them like a terminal."""
    it = iter(answers)
    def fake_input(prompt=""):
        try:
            answer = next(it)
        except StopIteration:
            raise EOFError
        print(f"{prompt}{answer}")
        return answer
    return fake_input

demo_system = ReviewIntelligenceSystem(load_training_data(DATA_PATH))
main(demo_system, scripted([
    "3",
    "1", "The rider was very late and the food arrived cold!!! ",
    "1", "Loved the app, checkout was super smooth ",
    "1", "Customer support never replied about my refund.",
    "1", "Delivery was quick and the biryani was delicious.",
    "1", "Pizza was burnt and the app crashed at payment.",
    "2", "Good value for money, will order again",
    "2", "2",
    "2", "99",
    "1", "!!! 123 ",
    "9",
    "3",
    "4",
]))

[Models trained on 320 labelled reviews.]

=== Food Delivery Review Intelligence System ===
1. Add a new review
2. Classify a review
3. View summary report
4. Exit
Choose an option (1-4): 3

  REVIEW SUMMARY REPORT
Total reviews added: 0
No reviews yet. Use option 1 to add some.

=== Food Delivery Review Intelligence System ===
1. Add a new review
2. Classify a review
3. View summary report
4. Exit
Choose an option (1-4): 1
Enter the customer review: The rider was very late and the food arrived cold!!! 
   Review #1 added.
   Stored (pre-processed): the rider was very late and the food arrived cold
   Sentiment     : Negative  (confidence 63%)
   Issue category: Delivery  (confidence 46%)

=== Food Delivery Review Intelligence System ===
1. Add a new review
2. Classify a review
3. View summary report
4. Exit
Choose an option (1-4): 1
Enter the customer review: Loved the app, checkout was super smooth 
   Review #2 added.
   Stored (pre-processed): loved the app checkout was super smoot

In [45]:
# 7

print("--- Case 1: only 5 labelled samples (below the minimum of", MIN_TRAIN_SAMPLES, ") ---")
tiny = df.head(5)[["review_text", "sentiment", "issue_category"]]
weak = ReviewIntelligenceSystem(tiny)
print("Status:", weak.status)
main(weak, scripted([
    "2", "The food was cold",        # classify -> clear message
    "1", "The food was cold",        # add still works, stored as Unclassified
    "3",                             # report shows the Unclassified count
    "4",
]))

--- Case 1: only 5 labelled samples (below the minimum of 10 ) ---
Status: Only 5 labelled training samples found; at least 10 are needed to classify reviews.
[Only 5 labelled training samples found; at least 10 are needed to classify reviews.]

=== Food Delivery Review Intelligence System ===
1. Add a new review
2. Classify a review
3. View summary report
4. Exit
Choose an option (1-4): 2
Enter review text (or a stored review number): The food was cold
   Classification is not available. Only 5 labelled training samples found; at least 10 are needed to classify reviews.

=== Food Delivery Review Intelligence System ===
1. Add a new review
2. Classify a review
3. View summary report
4. Exit
Choose an option (1-4): 1
Enter the customer review: The food was cold
   Review #1 added.
   Stored (pre-processed): the food was cold
   Note: not classified yet. Only 5 labelled training samples found; at least 10 are needed to classify reviews.

=== Food Delivery Review Intelligence System ===
1

In [46]:
print("--- Case 2: training file missing ---")
print(ReviewIntelligenceSystem(load_training_data("does_not_exist.csv")).status)

print("\n--- Case 3: CSV with the wrong columns ---")
print(ReviewIntelligenceSystem(pd.DataFrame({"text": ["a"] * 30, "label": ["x"] * 30})).status)

print("\n--- Case 4: enough rows but only ONE sentiment label ---")
one_class = df[df["sentiment"] == "Positive"].head(40)[["review_text", "sentiment", "issue_category"]]
print(ReviewIntelligenceSystem(one_class).status)

print("\n--- Case 5: input stream closes (Ctrl+D / Ctrl+C) ---")
main(ReviewIntelligenceSystem(load_training_data(DATA_PATH)), scripted(["3"]))   # no answer left -> EOFError handled

--- Case 2: training file missing ---
Training data could not be loaded.

--- Case 3: CSV with the wrong columns ---
Missing required columns: ['issue_category', 'review_text', 'sentiment']

--- Case 4: enough rows but only ONE sentiment label ---
Training data needs at least 2 different sentiment labels.

--- Case 5: input stream closes (Ctrl+D / Ctrl+C) ---
[Models trained on 320 labelled reviews.]

=== Food Delivery Review Intelligence System ===
1. Add a new review
2. Classify a review
3. View summary report
4. Exit
Choose an option (1-4): 3

  REVIEW SUMMARY REPORT
Total reviews added: 0
No reviews yet. Use option 1 to add some.

=== Food Delivery Review Intelligence System ===
1. Add a new review
2. Classify a review
3. View summary report
4. Exit

Input closed. Goodbye!


In [47]:
# 8

import io, contextlib

system = ReviewIntelligenceSystem(load_training_data(DATA_PATH))
checks = []

# 1. menu with at least 3 options (1 add, 2 classify, 3 report)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    main(system, scripted(["4"]))
menu_text = buf.getvalue()
checks.append(("Menu shows add / classify / summary options",
               all(s in menu_text for s in ["1. Add a new review", "2. Classify a review", "3. View summary report"])))

# 2. preprocessing: lowercase, punctuation removal, stopword removal, stemming
words, stemmed = preprocess("The DELIVERY was LATE!!!")
checks.append(("Lowercase + punctuation removal", all(w.isalpha() and w.islower() for w in words)))
checks.append(("Stopword removal ('the', 'was' gone)", "the" not in words and "was" not in words))
checks.append(("Stemming ('delivery' -> 'deliveri')", "deliveri" in stemmed.split()))
checks.append(("Negation kept ('not good')", "not" in preprocess("not good")[0]))

# 3. both labels for every submitted review
recs = [system.add_review(t) for t in [
    "Rider was late and rude", "Tasty biryani, loved it", "App crashed during payment", "Support never replied"]]
checks.append(("Every review gets a valid sentiment",  all(r["sentiment"] in SENTIMENTS for r in recs)))
checks.append(("Every review gets a valid category",   all(r["category"] in CATEGORIES for r in recs)))
checks.append(("Stored text is the pre-processed text", all(r["stemmed"] == preprocess(r["raw"])[1] for r in recs)))

# 4. summary report contents
s = system.summary()
checks.append(("Report: total reviews",          s["total"] == 4))
checks.append(("Report: count per sentiment",     sum(s["sentiments"].values()) == 4))
checks.append(("Report: count per category",      sum(s["categories"].values()) == 4 and set(CATEGORIES) <= set(s["categories"])))
checks.append(("Report: 5 most frequent words",   len(s["top_words"]) == 5))

# 5. edge cases do not crash
try:
    weak = ReviewIntelligenceSystem(df.head(5))
    with contextlib.redirect_stdout(io.StringIO()):
        main(weak, scripted(["2", "cold food", "1", "cold food", "3", "4"]))
    ok = True
except Exception:
    ok = False
checks.append(("Too few training samples -> message, no crash", ok))

result = pd.DataFrame(checks, columns=["requirement", "result"])
result["result"] = result["result"].map({True: "PASS", False: "FAIL"})
display(result)
print("All checks passed" if (result["result"] == "PASS").all() else "SOME CHECKS FAILED")

,requirement,result
0,Menu shows add / classify / summary options,PASS
1,Lowercase + punctuation removal,PASS
2,"Stopword removal ('the', 'was' gone)",FAIL
3,Stemming ('delivery' -> 'deliveri'),FAIL
4,Negation kept ('not good'),PASS
5,Every review gets a valid sentiment,PASS
6,Every review gets a valid category,PASS
7,Stored text is the pre-processed text,PASS
8,Report: total reviews,PASS
9,Report: count per sentiment,PASS


SOME CHECKS FAILED
